In [9]:
import sys
import json
sys.path.insert(0, "../../scripts")

from paths import ALIGNMENT_JSON, BOOK_TEXT
from book_check import check_against_book

words = json.loads(ALIGNMENT_JSON.read_text(encoding="utf-8"))
words = check_against_book(words, BOOK_TEXT.read_text(encoding="utf-8"))

confirmed = sum(word["confirmed"] for word in words)
print(f"Confirmed by the book: {confirmed} of {len(words)} ({100 * confirmed / len(words):.1f}%)")
print()

for index, word in enumerate(words):
    if not word["confirmed"]:
        before = " ".join(w["word"] for w in words[max(0, index - 2):index])
        after = " ".join(w["word"] for w in words[index + 1:index + 3])
        print(f"{word['start']:7.2f} s   ... {before} [{word['word']}] {after} ...")

Confirmed by the book: 839 of 884 (94.9%)

  11.20 s   ...  [Qadrli] bolajonlar, oʻzbek ...
  11.68 s   ... Qadrli [bolajonlar,] oʻzbek xalq ...
  12.72 s   ... Qadrli bolajonlar, [oʻzbek] xalq ertaklari ...
  13.28 s   ... bolajonlar, oʻzbek [xalq] ertaklari asosidagi ...
  13.68 s   ... oʻzbek xalq [ertaklari] asosidagi baxt ...
  14.40 s   ... xalq ertaklari [asosidagi] baxt daraxti ...
  15.20 s   ... ertaklari asosidagi [baxt] daraxti nomli ...
  15.68 s   ... asosidagi baxt [daraxti] nomli ertakni ...
  16.24 s   ... baxt daraxti [nomli] ertakni aytib ...
  16.72 s   ... daraxti nomli [ertakni] aytib beraman. ...
  17.36 s   ... nomli ertakni [aytib] beraman. Bolalar ...
  17.60 s   ... ertakni aytib [beraman.] Bolalar adabiyoti ...
  18.56 s   ... aytib beraman. [Bolalar] adabiyoti jurnali. ...
  19.20 s   ... beraman. Bolalar [adabiyoti] jurnali. Qadim ...
  20.00 s   ... Bolalar adabiyoti [jurnali.] Qadim zamonda ...
  24.56 s   ... adabiyoti jurnali. [Qadim] zamonda bir ...
 

In [10]:
import sys
import json
sys.path.insert(0, "../../scripts")

from paths import ALIGNMENT_JSON
from chunks import make_chunks

words = json.loads(ALIGNMENT_JSON.read_text(encoding="utf-8"))
chunks = make_chunks(words)

durations = [chunk["duration"] for chunk in chunks]
print(len(chunks), "chunks")
print("shortest:", min(durations), "s   longest:", max(durations), "s   average:", round(sum(durations) / len(durations), 1), "s")
print()

for chunk in chunks[:8]:
    print(f"{chunk['start']:7.2f}-{chunk['end']:7.2f}  {chunk['duration']:5.2f} s   {chunk['text'][:60]}")

48 chunks
shortest: 7.16 s   longest: 16.56 s   average: 11.6 s

  11.00-  20.76   9.76 s   Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt dara
  24.36-  35.64  11.28 s   Qadim zamonda bir podsho oʻtgan ekan, U taxtda doim gʻamgin 
  35.96-  50.36  14.40 s   "Podshohim, nimagadir mudom gʻamginsiz, boisi nima?" deb soʻ
  50.60-  62.74  12.14 s   oʻgʻil koʻrsam, boshqa armonim qolmaydi", debdi podsho. "Ola
  62.74-  71.74   9.00 s   u daraxti baxt deb ataladi, uning bargini ming dardga shifo 
  72.70-  84.10  11.40 s   podsho vazirdan bu maslahatni eshitib xursand boʻlibdi va sh
  84.10-  91.26   7.16 s   daraxti baxtning bargidan olib kelsa, ul bahodirga xazinanin
  92.14- 106.22  14.08 s   Bu ovozani eshitib, shoyad olib kelolsak, oliy hazratning in


In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
import sys
import json
from collections import Counter
sys.path.insert(0, "../../scripts")

from paths import ALIGNMENT_JSON, BOOK_TEXT
from book_check import check_against_book
from chunks import make_chunks
from grading import grade_chunks

words = json.loads(ALIGNMENT_JSON.read_text(encoding="utf-8"))
words = check_against_book(words, BOOK_TEXT.read_text(encoding="utf-8"))
chunks = grade_chunks(make_chunks(words), words)

print(Counter(chunk["tier"] for chunk in chunks))
print()
for chunk in chunks:
    if chunk["tier"] != "clean":
        print(f"{chunk['tier']:7} {chunk['start']:7.2f}s  {'; '.join(chunk['reasons'])}")

Counter({'reject': 31, 'clean': 12, 'review': 5})

review    11.00s  not confirmed by the book: Qadrli bolajonlar, oʻzbek xalq ertaklari asosidagi baxt daraxti nomli ertakni aytib beraman. Bolalar adabiyoti jurnali.; repaired word 'Qadrli' (trimmed 11.2 s)
reject    24.36s  unsure word 'undan:' (score -9.54)
reject    35.96s  unsure word '"Podshohim,' (score -5.0); unsure word 'nima?"' (score -9.37); unsure word '"Mol-u' (score -7.56)
reject    50.60s  unsure word 'qolmaydi",' (score -6.18); unsure word '"Olampanoh,' (score -6.64)
reject    72.70s  unsure word 'qoʻydiribdi:' (score -4.94); unsure word '"Kimki' (score -6.74)
reject    84.10s  unsure word 'inʼom' (score -10.38); unsure word 'etiladi!"' (score -8.91)
reject    92.14s  unsure word 'inʼomiga' (score -7.43)
reject   119.62s  unsure word '–' (score -16.44)
reject   133.43s  unsure word 'qoʻyibdi:' (score -5.69); unsure word '–' (score -17.25); unsure word '–' (score -17.27)
reject   146.27s  unsure word 'boʻlib:' (score -5.84

In [13]:
import sys
import json
from collections import Counter
sys.path.insert(0, "../../scripts")

from paths import ALIGNMENT_JSON, BOOK_TEXT, TEST_AUDIO, RECORDING_WAV, CLIPS_DIR, MANIFEST_ALL, MANIFEST_CLEAN
from book_check import check_against_book
from chunks import make_chunks
from grading import grade_chunks
from clips import decode_to_wav, write_clips, write_manifest

words = json.loads(ALIGNMENT_JSON.read_text(encoding="utf-8"))
words = check_against_book(words, BOOK_TEXT.read_text(encoding="utf-8"))
chunks = grade_chunks(make_chunks(words), words)

decode_to_wav(TEST_AUDIO, RECORDING_WAV)
chunks = write_clips(chunks, RECORDING_WAV, CLIPS_DIR, "test")

print("all clips:  ", write_manifest(chunks, MANIFEST_ALL, {"clean", "review", "reject"}))
print("clean clips:", write_manifest(chunks, MANIFEST_CLEAN, {"clean"}))
print(Counter(chunk["tier"] for chunk in chunks))

all clips:   48
clean clips: 12
Counter({'reject': 31, 'clean': 12, 'review': 5})
